In [2]:
!pip install -q FlagEmbedding qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.5/250.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.8/947.8 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.8 MB/s eta 0:00:00


In [3]:
import os

from FlagEmbedding import BGEM3FlagModel
from qdrant_client import QdrantClient, models

COLLECTION_NAME = "oracle_docs_321"


In [4]:
def _secret(name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ[name]


client = QdrantClient(url=_secret("QDRANT_URL"), api_key=_secret("QDRANT_API_KEY"))

import torch
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=torch.cuda.is_available())


def sparse_to_qdrant(w: dict) -> models.SparseVector:
    return models.SparseVector(
        indices=[int(k) for k in w.keys()],
        values=[float(v) for v in w.values()],
    )


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [9]:
def _filter(doc=None, types=None):
    must = []
    if doc:
        must.append(models.FieldCondition(key="doc", match=models.MatchValue(value=doc)))
    if types:
        must.append(models.FieldCondition(key="type", match=models.MatchAny(any=list(types))))
    return models.Filter(must=must) if must else None


def hybrid_search(query: str, top_k: int = 8, mode: str = "hybrid",
                  doc: str | None = None, types: list[str] | None = None,
                  prefetch_k: int = 50):
    """mode: 'hybrid' (default), 'dense' or 'sparse' - the last two are only
    for comparing against hybrid when you evaluate."""
    out = model.encode([query], max_length=512, return_dense=True,
                       return_sparse=True, return_colbert_vecs=False)
    dense = out["dense_vecs"][0].tolist()
    sparse = sparse_to_qdrant(out["lexical_weights"][0])
    flt = _filter(doc, types)

    if mode == "dense":
        res = client.query_points(COLLECTION_NAME, query=dense, using="dense",
                                  query_filter=flt, limit=top_k, with_payload=True)
    elif mode == "sparse":
        res = client.query_points(COLLECTION_NAME, query=sparse, using="sparse",
                                  query_filter=flt, limit=top_k, with_payload=True)
    else:
        res = client.query_points(
            COLLECTION_NAME,
            prefetch=[
                models.Prefetch(query=dense, using="dense", filter=flt, limit=prefetch_k),
                models.Prefetch(query=sparse, using="sparse", filter=flt, limit=prefetch_k),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=top_k,
            with_payload=True,
        )
    return res.points



In [10]:
def fetch_units(doc: str, indices):
    """Fetch specific units of one document, in reading order."""
    indices = sorted(set(indices))
    pts, _ = client.scroll(
        COLLECTION_NAME,
        scroll_filter=models.Filter(must=[
            models.FieldCondition(key="doc", match=models.MatchValue(value=doc)),
            models.FieldCondition(key="unit_index", match=models.MatchAny(any=indices)),
        ]),
        limit=len(indices),
        with_payload=True,
        with_vectors=False,
    )
    return sorted(pts, key=lambda p: p.payload["unit_index"])


def search_with_context(query: str, top_k: int = 5, radius: int = 5, **kw):
    """Hybrid search, then attach the units before/after each hit so the LLM
    sees the surrounding text. Overlapping windows are merged."""
    hits = hybrid_search(query, top_k=top_k, **kw)
    seen, results = set(), []
    for h in hits:
        d, i = h.payload["doc"], h.payload["unit_index"]
        if (d, i) in seen:                  # already covered by a better hit's window
            continue
        window = fetch_units(d, range(max(0, i - radius), i + radius + 1))
        seen.update((d, p.payload["unit_index"]) for p in window)
        results.append({
            "doc": d,
            "pages": sorted({p.payload.get("page") for p in window if p.payload.get("page") is not None}),
            "score": h.score,
            "hit_type": h.payload["type"],
            "text": "\n\n".join(p.payload["content"] for p in window),
        })
    return results


In [13]:
from FlagEmbedding import FlagReranker

reranker = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=torch.cuda.is_available())

def rerank(query, hits, keep=5):
    scores = reranker.compute_score(
        [[query, h.payload["content"]] for h in hits], normalize=True
    )
    ranked = sorted(zip(hits, scores), key=lambda x: x[1], reverse=True)
    return ranked[:keep]   # (hit, rerank_score) pairs — not just hits

q = "How to generate B2B tokens"
hits = hybrid_search(q, top_k=3)
reranked = rerank(q, hits)

for h, score in reranked:
    print(f"\n[{h.payload['doc']} p.{h.payload.get('page')}] rerank_score={score:.3f}")
    print(h.payload["content"][:400])

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


[Oracle_IDAM_RAG_Optimized p.8] rerank_score=0.960
B2B OAuth Client Procedure
Open the SIT OAM Oauth 12c IndianOilB2BUserProfileServiceDomain collection in the IDAM Postman Collection.
Use the Admin API request.
Create a new request according to the requirement (Duplicate any existing file/request ,then modify the duplicated value by changing its name, Secret and Client)
Click Send and generate the token.
RAG KEYWORDS: B2B, SIT, UPGRADE, OAM OAuth

[oracle-fusion-middleware-developing-applications-oracle-access-management p.44] rerank_score=0.009
Create a string tokenizer named tok2 and parse nameValue using the equal character (=) as the delimiter. In this manner, pn=vn breaks down into the tokens pn and vn.

[understanding-interoperability-and-compatibility p.12] rerank_score=0.000
DB21085I Instance "DB2? uses "32? bits and DB2 code release "SQL09011? with level identifier "01020107?. Informational tokens are "DB2 v9.1.100.129?, "s061104?, "WR21374?, and Fix Pack "1?. Product is ins